# 07 — Advanced traditional forecasting models

This notebook reproduces the **advanced traditional** model family used for greenhouse air-temperature and relative-humidity forecasting. Five model families are evaluated: ARIMA, SARIMA with explicit 24-hour differencing, exact SARIMA, structural state-space models, and an approximate ARTFIMA implementation based on tempered fractional differencing followed by ARMA.

All candidates use the same forecast-origin eligibility rule, chronological partitions, five temporal resolutions, and four forecast horizons defined in notebooks `03`–`06`. The common 24-hour window is the effective-history requirement; seasonal and fractional transformations may additionally read the pre-window lags required to construct that transformed window. Candidate selection is performed separately for each resolution and target using validation data only. The primary criterion is mean validation NRMSE across the four horizons; mean validation R² is the secondary criterion. The test partition is evaluated only after the configuration has been selected.

Full execution is computationally intensive, especially exact SARIMA. Run notebooks `01`–`06` first.


## Dependency note

This notebook requires `statsmodels`, a Parquet engine, and `tqdm`. If a dependency is missing, run the following command in a separate Jupyter cell, restart the kernel, and execute the notebook again:

```python
%pip install "statsmodels>=0.14,<0.15" "pyarrow>=15" tqdm
```


In [ ]:
from pathlib import Path
import importlib.util
import json
import math
import platform
import time
import warnings

if importlib.util.find_spec("statsmodels") is None:
    raise ImportError(
        "statsmodels is not installed. Run `%pip install \"statsmodels>=0.14,<0.15\"`, "
        "restart the kernel, and execute the notebook again."
    )
if importlib.util.find_spec("tqdm") is None:
    raise ImportError(
        "tqdm is not installed. Run `%pip install tqdm`, restart the kernel, "
        "and execute the notebook again."
    )

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.structural import UnobservedComponents
from statsmodels.tsa.stattools import acf
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

print(f"statsmodels: {statsmodels.__version__}")


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "resolutions").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run notebook 03 first and keep the standard folder structure."
    )


PROJECT_ROOT = find_project_root()
RESOLUTION_DIR = PROJECT_ROOT / "data" / "processed" / "resolutions"
INDEX_DIR = PROJECT_ROOT / "data" / "processed" / "effective_indices"
RESULTS_DIR = PROJECT_ROOT / "results" / "advanced_traditional"
PREDICTION_DIR = RESULTS_DIR / "predictions"
MODEL_DIR = PROJECT_ROOT / "models" / "advanced_traditional"
FIGURE_DIR = PROJECT_ROOT / "figures" / "advanced_traditional"
METADATA_DIR = PROJECT_ROOT / "metadata"

for directory in [RESULTS_DIR, PREDICTION_DIR, MODEL_DIR, FIGURE_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {platform.python_version()} | Platform: {platform.platform()}")


## Experimental configuration

`EXECUTION_MODE = "full"` reproduces the complete experiment. `"smoke_test"` is provided only to verify the software pipeline: it uses the 60-minute resolution, a reduced catalog, and at most 20 origins per split. Smoke-test results must not be used in the manuscript.

Exact SARIMA is restricted to seasonal periods no longer than 72 steps. Therefore it is evaluated at 20-, 30-, and 60-minute resolutions, but not at 4 or 12 minutes.


In [ ]:
EXECUTION_MODE = "full"  # Use "smoke_test" only for a quick technical check.

RESOLUTIONS = [4, 12, 20, 30, 60]
TARGETS = ["temperature", "relative_humidity"]
HORIZONS_MINUTES = [60, 120, 240, 480]
HISTORY_HOURS = 24
EXACT_SARIMA_MAX_PERIOD = 72
FRACTIONAL_WEIGHT_TOLERANCE = 1e-6
MAXIMUM_FRACTIONAL_LAGS = 500
MODEL_FAMILIES = [
    "ARIMA",
    "SARIMA_DAILY_DIFFERENCE",
    "SARIMA_EXACT",
    "STATE_SPACE",
    "ARTFIMA_APPROX",
]

BASE_CANDIDATES = {
    "ARIMA": [
        {"candidate_id": 1, "order": [1, 0, 0], "trend": "c"},
        {"candidate_id": 2, "order": [2, 0, 0], "trend": "c"},
        {"candidate_id": 3, "order": [1, 0, 1], "trend": "c"},
        {"candidate_id": 4, "order": [1, 1, 1], "trend": "t"},
    ],
    "SARIMA_DAILY_DIFFERENCE": [
        {"candidate_id": 1, "order": [1, 0, 0], "seasonal_difference": 1, "trend": "c"},
        {"candidate_id": 2, "order": [2, 0, 0], "seasonal_difference": 1, "trend": "c"},
        {"candidate_id": 3, "order": [1, 0, 1], "seasonal_difference": 1, "trend": "c"},
    ],
    "SARIMA_EXACT": [
        {"candidate_id": 1, "order": [1, 0, 0], "seasonal_order_base": [1, 0, 0], "trend": "c"},
        {"candidate_id": 2, "order": [1, 0, 1], "seasonal_order_base": [0, 1, 1], "trend": "c"},
    ],
    "STATE_SPACE": [
        {"candidate_id": 1, "autoregressive": 1, "harmonics": 3, "level": "local level", "stochastic_level": True},
        {"candidate_id": 2, "autoregressive": 1, "harmonics": 3, "level": "local linear trend", "stochastic_level": True, "stochastic_trend": True},
        {"candidate_id": 3, "autoregressive": 2, "harmonics": 6, "level": "local linear trend", "stochastic_level": True, "stochastic_trend": True},
    ],
    "ARTFIMA_APPROX": [
        {"candidate_id": 1, "arma_order": [1, 0], "fractional_d": 0.15, "tempering_lambda": 0.005},
        {"candidate_id": 2, "arma_order": [1, 1], "fractional_d": 0.25, "tempering_lambda": 0.01},
        {"candidate_id": 3, "arma_order": [2, 0], "fractional_d": 0.35, "tempering_lambda": 0.01},
        {"candidate_id": 4, "arma_order": [2, 1], "fractional_d": 0.25, "tempering_lambda": 0.02},
    ],
}

if EXECUTION_MODE == "full":
    ACTIVE_RESOLUTIONS = RESOLUTIONS
    ACTIVE_CANDIDATES = BASE_CANDIDATES
    MAX_ORIGINS_PER_SPLIT = None
elif EXECUTION_MODE == "smoke_test":
    ACTIVE_RESOLUTIONS = [60]
    ACTIVE_CANDIDATES = {
        "ARIMA": BASE_CANDIDATES["ARIMA"][:1],
        "SARIMA_DAILY_DIFFERENCE": BASE_CANDIDATES["SARIMA_DAILY_DIFFERENCE"][:1],
        "STATE_SPACE": BASE_CANDIDATES["STATE_SPACE"][:1],
        "ARTFIMA_APPROX": BASE_CANDIDATES["ARTFIMA_APPROX"][:1],
    }
    MAX_ORIGINS_PER_SPLIT = 20
else:
    raise ValueError("EXECUTION_MODE must be 'full' or 'smoke_test'.")

configuration = {
    "execution_mode": EXECUTION_MODE,
    "temporal_resolutions_minutes": RESOLUTIONS,
    "targets": TARGETS,
    "forecast_horizons_minutes": HORIZONS_MINUTES,
    "history_hours": HISTORY_HOURS,
    "model_families": MODEL_FAMILIES,
    "candidate_catalog": BASE_CANDIDATES,
    "exact_sarima_max_period": EXACT_SARIMA_MAX_PERIOD,
    "artfima_definition": "Tempered fractional differencing followed by ARMA; explicitly reported as an approximation.",
    "selection_rule": "Lowest mean validation NRMSE across horizons; highest mean validation R2 as secondary criterion.",
    "normalization_rule": "RMSE divided by the target standard deviation in the corresponding fitting partition.",
    "forecast_rule": "Parameters are estimated on the fitting partition and applied without refitting at each forecast origin. Seasonal and fractional transformations may read only the pre-window lags needed to construct the common 24-hour transformed history.",
    "test_policy": "Test is not used for family or candidate selection.",
    "progress_reporting": "Nested tqdm bars report candidate fits and forecast origins in validation and final test evaluation.",
}
(METADATA_DIR / "07_advanced_traditional_configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)
configuration


## Load the common forecast origins

The effective-origin files generated earlier are authoritative. The notebook verifies the historical counts before fitting any model. Missing values inside a model history are resolved causally by forward fill and then by a median estimated from the fitting partition.


In [ ]:
EXPECTED_COUNTS = {
    4: {"train": 1053, "validation": 1446, "test": 2742},
    12: {"train": 104, "validation": 135, "test": 378},
    20: {"train": 146, "validation": 239, "test": 556},
    30: {"train": 104, "validation": 142, "test": 350},
    60: {"train": 66, "validation": 58, "test": 211},
}


def load_inputs(resolution_minutes):
    data_path = RESOLUTION_DIR / f"greenhouse_{resolution_minutes}min.csv"
    index_path = INDEX_DIR / f"effective_indices_{resolution_minutes}min.csv"
    if not data_path.exists() or not index_path.exists():
        raise FileNotFoundError(
            f"Missing inputs for {resolution_minutes} min. Run notebook 03 first."
        )
    data = pd.read_csv(data_path, parse_dates=["timestamp"])
    data = data.sort_values("timestamp").reset_index(drop=True)
    origins = pd.read_csv(index_path, parse_dates=["origin_timestamp"])
    return data, origins


def causal_series(data, target, fitting_splits):
    values = data[target].astype(float).ffill()
    fit_mask = data["split"].isin(fitting_splits)
    fit_median = float(values.loc[fit_mask].median())
    return values.fillna(fit_median).to_numpy(dtype=float), fit_median


datasets = {}
origins_by_resolution = {}
audit_rows = []
for resolution in RESOLUTIONS:
    data, origins = load_inputs(resolution)
    datasets[resolution] = data
    origins_by_resolution[resolution] = origins
    row = {
        "resolution": f"{resolution}min",
        "dataset_rows": len(data),
        "seasonal_steps_24h": 24 * 60 // resolution,
    }
    for split, expected in EXPECTED_COUNTS[resolution].items():
        observed = int(origins["split"].eq(split).sum())
        row[f"{split}_origins"] = observed
        if observed != expected:
            raise AssertionError(
                f"{resolution} min {split}: observed {observed}, expected {expected}."
            )
    audit_rows.append(row)

sample_audit = pd.DataFrame(audit_rows)
sample_audit.to_csv(RESULTS_DIR / "02_sample_audit.csv", index=False)
display(sample_audit)


## Candidate catalog

The catalog is fixed before validation. Exact SARIMA candidates are omitted automatically when the 24-hour seasonal period exceeds 72 steps.


In [ ]:
def candidates_for_resolution(resolution_minutes):
    seasonal_steps = 24 * 60 // resolution_minutes
    rows = []
    for family, candidates in ACTIVE_CANDIDATES.items():
        if family == "SARIMA_EXACT" and seasonal_steps > EXACT_SARIMA_MAX_PERIOD:
            continue
        for candidate in candidates:
            parameters = {k: v for k, v in candidate.items() if k != "candidate_id"}
            rows.append({
                "resolution": f"{resolution_minutes}min",
                "resolution_minutes": resolution_minutes,
                "family": family,
                "candidate_id": candidate["candidate_id"],
                "parameters": json.dumps(parameters, sort_keys=True),
            })
    return rows


candidate_catalog = pd.DataFrame([
    row
    for resolution in ACTIVE_RESOLUTIONS
    for row in candidates_for_resolution(resolution)
])
candidate_catalog.to_csv(RESULTS_DIR / "01_candidate_catalog.csv", index=False)
display(candidate_catalog.groupby(["resolution", "family"]).size().rename("candidates").reset_index())


## Model implementations

`SARIMA_DAILY_DIFFERENCE` models the explicit transformation

\[
z_t = y_t - y_{t-s},
\]

where (s) is the number of samples in 24 hours. Forecasts are mapped back to the original scale recursively. `ARTFIMA_APPROX` uses finite tempered fractional-difference weights and an ARMA model; it is not presented as an exact ARTFIMA estimator.


In [ ]:
def fractional_weights(d, tempering_lambda, maximum_lags, tolerance):
    weights = [1.0]
    for lag in range(1, maximum_lags + 1):
        untempered = -weights[-1] * (d - lag + 1) / lag
        weight = untempered * math.exp(-tempering_lambda)
        weights.append(weight)
        if lag >= 20 and abs(weight) < tolerance:
            break
    return np.asarray(weights, dtype=float)


def fractional_difference(values, weights):
    values = np.asarray(values, dtype=float)
    output = np.empty(len(values), dtype=float)
    for index in range(len(values)):
        available = min(index + 1, len(weights))
        output[index] = np.dot(weights[:available], values[index - available + 1:index + 1][::-1])
    return output


def fit_candidate(values, family, parameters, seasonal_steps):
    values = np.asarray(values, dtype=float)
    started = time.perf_counter()

    if family == "ARIMA":
        result = ARIMA(
            values, order=tuple(parameters["order"]), trend=parameters["trend"]
        ).fit()
        auxiliary = {}

    elif family == "SARIMA_DAILY_DIFFERENCE":
        differenced = values[seasonal_steps:] - values[:-seasonal_steps]
        result = ARIMA(
            differenced, order=tuple(parameters["order"]), trend=parameters["trend"]
        ).fit()
        auxiliary = {}

    elif family == "SARIMA_EXACT":
        seasonal_base = tuple(parameters["seasonal_order_base"])
        seasonal_order = (*seasonal_base, seasonal_steps)
        result = SARIMAX(
            values,
            order=tuple(parameters["order"]),
            seasonal_order=seasonal_order,
            trend=parameters["trend"],
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False, maxiter=200)
        auxiliary = {"seasonal_order": seasonal_order}

    elif family == "STATE_SPACE":
        harmonics = min(int(parameters["harmonics"]), max(1, seasonal_steps // 2))
        result = UnobservedComponents(
            values,
            level=parameters["level"],
            autoregressive=int(parameters["autoregressive"]),
            stochastic_level=bool(parameters.get("stochastic_level", False)),
            stochastic_trend=bool(parameters.get("stochastic_trend", False)),
            freq_seasonal=[{"period": seasonal_steps, "harmonics": harmonics}],
        ).fit(disp=False, maxiter=200)
        auxiliary = {"harmonics_used": harmonics}

    elif family == "ARTFIMA_APPROX":
        maximum_lags = min(MAXIMUM_FRACTIONAL_LAGS, max(20, len(values) // 4))
        weights = fractional_weights(
            parameters["fractional_d"], parameters["tempering_lambda"],
            maximum_lags, FRACTIONAL_WEIGHT_TOLERANCE,
        )
        transformed = fractional_difference(values, weights)
        p, q = parameters["arma_order"]
        result = ARIMA(transformed, order=(int(p), 0, int(q)), trend="c").fit()
        auxiliary = {"fractional_weights": weights}

    else:
        raise ValueError(f"Unknown model family: {family}")

    return {
        "family": family,
        "parameters": parameters,
        "seasonal_steps": seasonal_steps,
        "result": result,
        "auxiliary": auxiliary,
        "fit_seconds": time.perf_counter() - started,
        "aic": float(getattr(result, "aic", np.nan)),
        "bic": float(getattr(result, "bic", np.nan)),
    }


def forecast_from_history(bundle, history, maximum_steps):
    family = bundle["family"]
    result = bundle["result"]
    seasonal_steps = bundle["seasonal_steps"]
    history = np.asarray(history, dtype=float)

    if family in {"ARIMA", "SARIMA_EXACT", "STATE_SPACE"}:
        applied = result.apply(history, refit=False)
        return np.asarray(applied.forecast(maximum_steps), dtype=float)

    if family == "SARIMA_DAILY_DIFFERENCE":
        differenced = history[seasonal_steps:] - history[:-seasonal_steps]
        applied = result.apply(differenced, refit=False)
        difference_forecast = np.asarray(applied.forecast(maximum_steps), dtype=float)
        reconstructed = list(history)
        forecasts = []
        for difference in difference_forecast:
            value = reconstructed[-seasonal_steps] + difference
            reconstructed.append(float(value))
            forecasts.append(float(value))
        return np.asarray(forecasts)

    if family == "ARTFIMA_APPROX":
        weights = bundle["auxiliary"]["fractional_weights"]
        transformed = fractional_difference(history, weights)[-seasonal_steps:]
        applied = result.apply(transformed, refit=False)
        transformed_forecast = np.asarray(applied.forecast(maximum_steps), dtype=float)
        reconstructed = list(history)
        forecasts = []
        for transformed_value in transformed_forecast:
            available = min(len(reconstructed), len(weights) - 1)
            past = np.asarray(reconstructed[-available:][::-1], dtype=float)
            value = transformed_value - np.dot(weights[1:available + 1], past)
            reconstructed.append(float(value))
            forecasts.append(float(value))
        return np.asarray(forecasts)

    raise ValueError(family)


## Forecasting and metric utilities

Model parameters are estimated once per fitting partition. At each forecast origin, the fitted parameters are applied without refitting. ARIMA and structural state-space models use the common 24-hour window directly. Seasonal and fractional transformations can read only the additional pre-window lags required to construct their 24-hour transformed history. No observation after the forecast origin is used.


In [ ]:
def select_origins(origins, split):
    selected = origins.loc[origins["split"].eq(split)].copy()
    if MAX_ORIGINS_PER_SPLIT is not None and len(selected) > MAX_ORIGINS_PER_SPLIT:
        positions = np.linspace(0, len(selected) - 1, MAX_ORIGINS_PER_SPLIT).round().astype(int)
        selected = selected.iloc[np.unique(positions)].copy()
    return selected


def predict_origins(
    bundle, data, origins, values, resolution, target, split,
    progress_description=None,
):
    selected = select_origins(origins, split)
    maximum_steps = max(HORIZONS_MINUTES) // resolution
    horizon_steps = {h: h // resolution for h in HORIZONS_MINUTES}
    rows = []
    started = time.perf_counter()

    origin_iterator = tqdm(
        selected.itertuples(index=False),
        total=len(selected),
        desc=progress_description or f"{split} forecast origins",
        unit="origin",
        leave=False,
        position=1,
        dynamic_ncols=True,
    )
    for origin in origin_iterator:
        start = int(origin.history_start_index)
        end = int(origin.history_end_index) + 1
        if bundle["family"] in {"SARIMA_DAILY_DIFFERENCE", "SARIMA_EXACT"}:
            context_start = max(0, start - bundle["seasonal_steps"])
        elif bundle["family"] == "ARTFIMA_APPROX":
            extra_lags = len(bundle["auxiliary"]["fractional_weights"]) - 1
            context_start = max(0, start - extra_lags)
        else:
            context_start = start
        history = values[context_start:end]
        forecast = forecast_from_history(bundle, history, maximum_steps)
        for horizon, steps in horizon_steps.items():
            target_index = int(origin.origin_index) + steps
            rows.append({
                "resolution": f"{resolution}min",
                "resolution_minutes": resolution,
                "family": bundle["family"],
                "candidate_id": bundle["parameters"].get("candidate_id"),
                "parameters": json.dumps({k: v for k, v in bundle["parameters"].items() if k != "candidate_id"}, sort_keys=True),
                "split": split,
                "target": target,
                "horizon_minutes": horizon,
                "origin_index": int(origin.origin_index),
                "origin_timestamp": origin.origin_timestamp,
                "target_timestamp": data.loc[target_index, "timestamp"],
                "observed": float(values[target_index]),
                "predicted": float(forecast[steps - 1]),
            })

    return pd.DataFrame(rows), time.perf_counter() - started


def metric_table(predictions, fitting_scale):
    group_columns = [
        "resolution", "resolution_minutes", "family", "candidate_id", "parameters",
        "split", "target", "horizon_minutes",
    ]
    rows = []
    for keys, group in predictions.groupby(group_columns, sort=False, observed=True):
        observed = group["observed"].to_numpy(dtype=float)
        predicted = group["predicted"].to_numpy(dtype=float)
        error = predicted - observed
        row = dict(zip(group_columns, keys))
        row.update({
            "n": len(group),
            "rmse": mean_squared_error(observed, predicted) ** 0.5,
            "mae": mean_absolute_error(observed, predicted),
            "bias": float(error.mean()),
            "r2": r2_score(observed, predicted),
        })
        row["nrmse"] = row["rmse"] / fitting_scale[(row["resolution_minutes"], row["target"], row["split"])]
        rows.append(row)
    return pd.DataFrame(rows)


## Phase A — validation-only candidate selection

Every candidate is fitted using training data. Forecast errors are then calculated at the common validation origins. Candidate failures are recorded rather than silently discarded.


In [ ]:
validation_prediction_frames = []
fit_rows = []
failure_rows = []
fitting_scale = {}

phase_a_total_runs = len(candidate_catalog) * len(TARGETS)
phase_a_progress = tqdm(
    total=phase_a_total_runs,
    desc="Phase A — validation candidates",
    unit="candidate",
    position=0,
    dynamic_ncols=True,
)

for resolution in ACTIVE_RESOLUTIONS:
    data = datasets[resolution]
    origins = origins_by_resolution[resolution]
    seasonal_steps = 24 * 60 // resolution
    training_mask = data["split"].eq("train")

    for target in TARGETS:
        values, fit_median = causal_series(data, target, ["train"])
        fitting_values = values[training_mask.to_numpy()]
        fitting_scale[(resolution, target, "validation")] = float(np.std(fitting_values, ddof=0))

        catalog_subset = candidate_catalog.loc[candidate_catalog["resolution_minutes"].eq(resolution)]
        for candidate in catalog_subset.itertuples(index=False):
            parameters = json.loads(candidate.parameters)
            parameters["candidate_id"] = int(candidate.candidate_id)
            run_label = (
                f"{resolution} min | {target} | {candidate.family} | "
                f"candidate {int(candidate.candidate_id)}"
            )
            phase_a_progress.set_description_str(run_label, refresh=True)
            run_started = time.perf_counter()
            run_status = "ok"
            try:
                bundle = fit_candidate(fitting_values, candidate.family, parameters, seasonal_steps)
                predictions, inference_seconds = predict_origins(
                    bundle, data, origins, values, resolution, target, "validation",
                    progress_description=f"Forecast origins | {run_label}",
                )
                validation_prediction_frames.append(predictions)
                fit_rows.append({
                    "resolution": f"{resolution}min",
                    "target": target,
                    "family": candidate.family,
                    "candidate_id": int(candidate.candidate_id),
                    "parameters": candidate.parameters,
                    "fit_partition": "train",
                    "fit_rows": len(fitting_values),
                    "fit_median": fit_median,
                    "fit_seconds": bundle["fit_seconds"],
                    "inference_seconds": inference_seconds,
                    "aic": bundle["aic"],
                    "bic": bundle["bic"],
                    "status": "ok",
                })
            except Exception as error:
                run_status = type(error).__name__
                failure_rows.append({
                    "resolution": f"{resolution}min",
                    "target": target,
                    "family": candidate.family,
                    "candidate_id": int(candidate.candidate_id),
                    "parameters": candidate.parameters,
                    "phase": "validation",
                    "error_type": type(error).__name__,
                    "error_message": str(error),
                })
            finally:
                phase_a_progress.set_postfix({
                    "status": run_status,
                    "last_min": f"{(time.perf_counter() - run_started) / 60:.1f}",
                }, refresh=False)
                phase_a_progress.update(1)

phase_a_progress.close()
if not validation_prediction_frames:
    raise RuntimeError("No validation candidate completed successfully.")

validation_predictions = pd.concat(validation_prediction_frames, ignore_index=True)
validation_metrics = metric_table(validation_predictions, fitting_scale)
validation_predictions.to_parquet(PREDICTION_DIR / "01_all_candidate_validation_predictions.parquet", index=False)
validation_metrics.to_csv(RESULTS_DIR / "04_candidate_validation_metrics_by_horizon.csv", index=False)

fit_summary = pd.DataFrame(fit_rows)
candidate_failures = pd.DataFrame(failure_rows, columns=[
    "resolution", "target", "family", "candidate_id", "parameters",
    "phase", "error_type", "error_message",
])
fit_summary.to_csv(RESULTS_DIR / "09_fit_summary.csv", index=False)
candidate_failures.to_csv(RESULTS_DIR / "05_candidate_failures.csv", index=False)

validation_summary = (
    validation_metrics.groupby(
        ["resolution", "resolution_minutes", "target", "family", "candidate_id", "parameters"],
        as_index=False, observed=True,
    )
    .agg(
        validation_nrmse=("nrmse", "mean"),
        validation_rmse=("rmse", "mean"),
        validation_r2=("r2", "mean"),
    )
)
validation_summary = validation_summary.merge(
    fit_summary[["resolution", "target", "family", "candidate_id", "aic", "bic", "fit_seconds"]],
    on=["resolution", "target", "family", "candidate_id"], how="left",
)
validation_summary.to_csv(RESULTS_DIR / "03_candidate_validation_summary.csv", index=False)
display(validation_summary.sort_values(["resolution_minutes", "target", "validation_nrmse"]).head(20))


In [ ]:
selection_columns = ["resolution_minutes", "target"]
selected_configurations = (
    validation_summary.sort_values(
        selection_columns + ["validation_nrmse", "validation_r2"],
        ascending=[True, True, True, False],
    )
    .groupby(selection_columns, as_index=False, sort=False)
    .head(1)
    .reset_index(drop=True)
)
selected_configurations["selection_rule"] = (
    "Lowest mean validation NRMSE; mean validation R2 as secondary criterion"
)
selected_configurations.to_csv(
    RESULTS_DIR / "07_selected_configuration_by_resolution_target.csv", index=False
)
display(selected_configurations)


## Phase B — final fitting and independent test evaluation

The selected configuration is refitted using `train + validation`. Test forecasts are then generated once. Selected test metrics and predictions are the only test products used by the subsequent hybrid and benchmark notebooks.


In [ ]:
test_prediction_frames = []
final_fit_rows = []

phase_b_progress = tqdm(
    total=len(selected_configurations),
    desc="Phase B — final refit and test",
    unit="model",
    position=0,
    dynamic_ncols=True,
)

for selected in selected_configurations.itertuples(index=False):
    resolution = int(selected.resolution_minutes)
    target = selected.target
    data = datasets[resolution]
    origins = origins_by_resolution[resolution]
    seasonal_steps = 24 * 60 // resolution
    parameters = json.loads(selected.parameters)
    parameters["candidate_id"] = int(selected.candidate_id)
    run_label = (
        f"{resolution} min | {target} | {selected.family} | "
        f"candidate {int(selected.candidate_id)}"
    )
    phase_b_progress.set_description_str(run_label, refresh=True)
    run_started = time.perf_counter()

    values, fit_median = causal_series(data, target, ["train", "validation"])
    final_mask = data["split"].isin(["train", "validation"]).to_numpy()
    fitting_values = values[final_mask]
    fitting_scale[(resolution, target, "test")] = float(np.std(fitting_values, ddof=0))

    bundle = fit_candidate(fitting_values, selected.family, parameters, seasonal_steps)
    predictions, inference_seconds = predict_origins(
        bundle, data, origins, values, resolution, target, "test",
        progress_description=f"Test origins | {run_label}",
    )
    test_prediction_frames.append(predictions)

    model_path = MODEL_DIR / f"{resolution}min" / target / f"{selected.family}_candidate_{int(selected.candidate_id)}.joblib"
    model_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, model_path)
    final_fit_rows.append({
        "resolution": f"{resolution}min",
        "target": target,
        "family": selected.family,
        "candidate_id": int(selected.candidate_id),
        "parameters": selected.parameters,
        "fit_partition": "train+validation",
        "fit_rows": len(fitting_values),
        "fit_median": fit_median,
        "fit_seconds": bundle["fit_seconds"],
        "inference_seconds": inference_seconds,
        "aic": bundle["aic"],
        "bic": bundle["bic"],
        "model_path": str(model_path.relative_to(PROJECT_ROOT)),
    })
    phase_b_progress.set_postfix({
        "last_min": f"{(time.perf_counter() - run_started) / 60:.1f}",
        "fit_min": f"{bundle['fit_seconds'] / 60:.1f}",
    }, refresh=False)
    phase_b_progress.update(1)

phase_b_progress.close()
test_predictions = pd.concat(test_prediction_frames, ignore_index=True)
test_metrics = metric_table(test_predictions, fitting_scale)
test_predictions.to_parquet(PREDICTION_DIR / "02_selected_advanced_traditional_test_predictions.parquet", index=False)
test_metrics.to_csv(RESULTS_DIR / "08_selected_advanced_traditional_test_metrics.csv", index=False)
pd.DataFrame(final_fit_rows).to_csv(RESULTS_DIR / "10_final_fit_summary.csv", index=False)
display(test_metrics)


## Long-memory and residual diagnostics

The long-memory table reports descriptive Hurst and GPH estimates after daily differencing. These diagnostics motivate the approximate ARTFIMA candidates but do not alter the predefined catalog. Ljung–Box tests are calculated from the selected out-of-sample test errors.


In [ ]:
def hurst_rs(values):
    values = np.asarray(values, dtype=float)
    block_sizes = np.unique(np.geomspace(16, max(17, len(values) // 4), 12).astype(int))
    log_size, log_rs = [], []
    for size in block_sizes:
        statistics = []
        for start in range(0, len(values) - size + 1, size):
            block = values[start:start + size]
            centered = block - block.mean()
            scale = block.std(ddof=1)
            if scale > 0:
                statistics.append(np.ptp(np.cumsum(centered)) / scale)
        if statistics and np.mean(statistics) > 0:
            log_size.append(np.log(size))
            log_rs.append(np.log(np.mean(statistics)))
    return float(np.polyfit(log_size, log_rs, 1)[0]) if len(log_size) >= 2 else np.nan


def gph_estimate(values):
    values = np.asarray(values, dtype=float)
    values = values - values.mean()
    n = len(values)
    m = max(10, int(np.sqrt(n)))
    frequencies = 2 * np.pi * np.arange(1, m + 1) / n
    transform = np.fft.fft(values)
    periodogram = (np.abs(transform[1:m + 1]) ** 2) / (2 * np.pi * n)
    x = np.log(4 * np.sin(frequencies / 2) ** 2)
    y = np.log(np.maximum(periodogram, np.finfo(float).tiny))
    slope, intercept, r_value, p_value, _ = stats.linregress(x, y)
    return {"gph_d": float(-slope), "gph_p_value": float(p_value), "gph_r2": float(r_value ** 2), "gph_m": m}


long_memory_rows = []
for resolution in ACTIVE_RESOLUTIONS:
    data = datasets[resolution]
    seasonal_steps = 24 * 60 // resolution
    for target in TARGETS:
        values, _ = causal_series(data, target, ["train"])
        training = values[data["split"].eq("train").to_numpy()]
        differenced = training[seasonal_steps:] - training[:-seasonal_steps]
        diagnostic = {
            "resolution": f"{resolution}min",
            "target": target,
            "seasonal_steps": seasonal_steps,
            "seasonally_differenced_n": len(differenced),
            "hurst_rs": hurst_rs(training),
            "acf_lag_1": float(acf(training, nlags=1, fft=True)[1]),
            "acf_lag_24h_after_difference": float(acf(differenced, nlags=min(seasonal_steps, len(differenced) - 1), fft=True)[-1]),
        }
        diagnostic.update(gph_estimate(differenced))
        long_memory_rows.append(diagnostic)

long_memory_diagnostics = pd.DataFrame(long_memory_rows)
long_memory_diagnostics.to_csv(RESULTS_DIR / "06_long_memory_diagnostics.csv", index=False)

residual_rows = []
for keys, group in test_predictions.groupby(
    ["resolution", "resolution_minutes", "target", "horizon_minutes", "family"],
    sort=False, observed=True,
):
    residuals = group["predicted"].to_numpy(dtype=float) - group["observed"].to_numpy(dtype=float)
    maximum_lag = min(max(1, 24 * 60 // int(keys[1])), max(1, len(residuals) // 5))
    lag = min(maximum_lag, len(residuals) - 1)
    lb = acorr_ljungbox(residuals, lags=[lag], return_df=True)
    residual_rows.append({
        "resolution": keys[0],
        "resolution_minutes": keys[1],
        "target": keys[2],
        "horizon_minutes": keys[3],
        "family": keys[4],
        "n": len(residuals),
        "error_acf_1": float(acf(residuals, nlags=1, fft=True)[1]) if len(residuals) > 2 else np.nan,
        "squared_error_acf_1": float(acf(residuals ** 2, nlags=1, fft=True)[1]) if len(residuals) > 2 else np.nan,
        "ljung_box_lag": lag,
        "ljung_box_statistic": float(lb["lb_stat"].iloc[0]),
        "ljung_box_p_value": float(lb["lb_pvalue"].iloc[0]),
        "residual_autocorrelation_flag": bool(lb["lb_pvalue"].iloc[0] < 0.05),
    })

residual_diagnostics = pd.DataFrame(residual_rows)
residual_diagnostics.to_csv(RESULTS_DIR / "11_residual_diagnostics.csv", index=False)
display(residual_diagnostics.head())


## Historical-selection audit

The table below records the configurations selected in the audited historical run. It is used as a reproducibility check, not as a forced selection rule. Minor numerical differences can occur across `statsmodels`, BLAS, and operating-system versions; any mismatch must be investigated and documented rather than overwritten.


In [ ]:
historical_selection = pd.DataFrame([
    [4, "temperature", "SARIMA_DAILY_DIFFERENCE", 2],
    [4, "relative_humidity", "SARIMA_DAILY_DIFFERENCE", 2],
    [12, "temperature", "SARIMA_DAILY_DIFFERENCE", 2],
    [12, "relative_humidity", "SARIMA_DAILY_DIFFERENCE", 3],
    [20, "temperature", "SARIMA_DAILY_DIFFERENCE", 2],
    [20, "relative_humidity", "SARIMA_EXACT", 2],
    [30, "temperature", "SARIMA_DAILY_DIFFERENCE", 2],
    [30, "relative_humidity", "SARIMA_EXACT", 2],
    [60, "temperature", "SARIMA_DAILY_DIFFERENCE", 2],
    [60, "relative_humidity", "STATE_SPACE", 3],
], columns=["resolution_minutes", "target", "historical_family", "historical_candidate_id"])

selection_audit = selected_configurations.merge(
    historical_selection, on=["resolution_minutes", "target"], how="left"
)
selection_audit["family_match"] = selection_audit["family"].eq(selection_audit["historical_family"])
selection_audit["candidate_match"] = selection_audit["candidate_id"].eq(selection_audit["historical_candidate_id"])
selection_audit["complete_match"] = selection_audit["family_match"] & selection_audit["candidate_match"]
selection_audit.to_csv(RESULTS_DIR / "12_historical_selection_audit.csv", index=False)
display(selection_audit)


## Diagnostic figure and reproducibility report


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for axis, target in zip(axes, TARGETS):
    subset = test_metrics.loc[test_metrics["target"].eq(target)]
    sns.lineplot(
        data=subset,
        x="horizon_minutes", y="rmse",
        hue="resolution", markers=True, dashes=False, ax=axis,
    )
    axis.set_title(target.replace("_", " ").title())
    axis.set_xlabel("Forecast horizon (min)")
    axis.set_ylabel("Test RMSE")
figure_path = FIGURE_DIR / "07_advanced_traditional_test_rmse.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

report = {
    "notebook": "07_advanced_traditional_models.ipynb",
    "execution_mode": EXECUTION_MODE,
    "python": platform.python_version(),
    "statsmodels": statsmodels.__version__,
    "candidate_rows": len(candidate_catalog),
    "successful_validation_fits": len(fit_summary),
    "candidate_failures": len(candidate_failures),
    "selected_configurations": len(selected_configurations),
    "validation_prediction_rows": len(validation_predictions),
    "test_prediction_rows": len(test_predictions),
    "test_metric_rows": len(test_metrics),
    "historical_selection_matches": int(selection_audit["complete_match"].sum()),
    "test_used_for_selection": False,
}
(METADATA_DIR / "07_advanced_traditional_report.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

display(pd.DataFrame({"item": report.keys(), "value": report.values()}))
print(f"Results: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Models: {MODEL_DIR.relative_to(PROJECT_ROOT)}")
print(f"Figure: {figure_path.relative_to(PROJECT_ROOT)}")


## Reproducibility note

- Keep `EXECUTION_MODE = "full"` for the scientific run.
- Do not alter the candidate catalog after inspecting validation or test results.
- Never use test metrics to choose a family or candidate.
- Retain the `ARTFIMA_APPROX` name because this notebook does not claim an exact ARTFIMA likelihood estimator.
- Run notebook `08_robust_residual_hybrid.ipynb` only after this notebook and notebooks `05`–`06` have completed in full mode.
